In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

import clickhouse_connect

# Show all rows
pd.set_option("display.max_rows", None)

# Show all columns
pd.set_option("display.max_columns", None)

# Make sure wide DataFrames don't wrap
pd.set_option("display.width", None)

# Show the full content of each column (no '...')
pd.set_option("display.max_colwidth", None)

In [2]:
client = clickhouse_connect.get_client(
    host='yaujulxk39.eastus2.azure.clickhouse.cloud',      # or server IP
    port=8443,             # HTTP port (default)
    username='default',
    password='~gZjRLjjOJh1i',
    database='Competitive_Analysis'
)

In [3]:
# ---- Query ----
df_hum_hist = client.query_df("""
    SELECT *
    FROM Competitive_Analysis.DRGHistoricalReviewInformation
    WHERE Client = 'Hum'
""")
df_hum_hist.shape

(76355, 291)

In [4]:
# -------------------------------------------------
# Initial shape of the dataset
# -------------------------------------------------
df_hum = df_hum_hist.copy()

print(df_hum.shape)  

df_hum = df_hum[df_hum['InitialDenialType'] == 'Substantive']
print(df_hum.shape) 

# -------------------------------------------------
# Identify valid Group_Type values
# -------------------------------------------------

group_type_unique = df_hum['Group_Type'].dropna().unique()
print(group_type_unique) 

# -------------------------------------------------
# Filter dataset to only valid Group_Type rows
# Use .copy() to avoid SettingWithCopyWarning
# -------------------------------------------------
df_hum_group_type = df_hum[df_hum['Group_Type'].isin(group_type_unique)].copy()
print(df_hum_group_type.shape) 

# -------------------------------------------------
# Calculate ADRG length
# -------------------------------------------------
df_hum_group_type.loc[:, 'len_adrg'] = (
    df_hum_group_type['ADRG'].astype(str).str.len()
)

print(df_hum_group_type['len_adrg'].value_counts())
print(df_hum_group_type['Group_Type'].value_counts())

# -------------------------------------------------
# Remove invalid DRG / Group_Type combinations
#   1) Group_Type = '01' and ADRG length = 4
#   2) Group_Type = '10' and ADRG length = 3
# -------------------------------------------------
mask_remove = (
    ((df_hum_group_type['Group_Type'] == '01') & (df_hum_group_type['len_adrg'] == 4)) |
    ((df_hum_group_type['Group_Type'] == '10') & (df_hum_group_type['len_adrg'] == 3))
)

df_hum_drg_rem = df_hum_group_type.loc[~mask_remove].copy()
print(df_hum_drg_rem.shape)

# -------------------------------------------------
# Create a copy of InitialDeterminationDate
# -------------------------------------------------
df_hum_drg_rem.loc[:, 'InitialDeterminationDate_copy'] = (
    df_hum_drg_rem['InitialDeterminationDate']
)

print(
    df_hum_drg_rem['Control_ID']
    .value_counts()
    .sort_values(ascending=False)
    .head()
)

# -------------------------------------------------
# Convert date column to datetime
# -------------------------------------------------
df_hum_drg_rem.loc[:, 'InitialDeterminationDate_copy'] = pd.to_datetime(
    df_hum_drg_rem['InitialDeterminationDate_copy'],
    errors='coerce'
)

# -------------------------------------------------
# Keep latest record per Control_ID
# -------------------------------------------------
df_sorted = df_hum_drg_rem.sort_values(
    'InitialDeterminationDate_copy',
    ascending=False
)

df_hum_latest = df_sorted.drop_duplicates(
    subset='Control_ID',
    keep='first'
).copy()

print(df_hum_latest.shape)

# -------------------------------------------------
# Normalize string columns
# -------------------------------------------------
df_hum_latest.loc[:, 'ADRG'] = (
    df_hum_latest['ADRG'].astype(str).str.strip()
)

df_hum_latest.loc[:, 'PRIM_DX'] = (
    df_hum_latest['PRIM_DX'].astype(str).str.strip().str.upper()
)

df_hum_latest.loc[:, 'InitialDeterminationStatus'] = (
    df_hum_latest['InitialDeterminationStatus']
    .astype(str)
    .str.strip()
    .str.upper()
)

# -------------------------------------------------
# Clean numeric fields
# -------------------------------------------------
df_hum_latest.loc[:, 'IDSavings'] = (
    pd.to_numeric(df_hum_latest['IDSavings'], errors='coerce').fillna(0)
)

df_hum_latest.loc[:, 'LOS'] = (
    pd.to_numeric(df_hum_latest['LOS'], errors='coerce').fillna(0)
)

df_hum_latest.loc[:, 'AGE'] = (
    pd.to_numeric(df_hum_latest['AGE'], errors='coerce')
)

# -------------------------------------------------
# Validate AGE values
# -------------------------------------------------
max_age = df_hum_latest['AGE'].max()

df_hum_latest = df_hum_latest[(
    df_hum_latest['AGE'].between(0, max_age))
].copy()
df_hum_latest.loc[:, 'AGE'] = df_hum_latest['AGE'].astype('Int64')
print(df_hum_latest.shape)

# -------------------------------------------------
# Validate LOS values
# -------------------------------------------------
df_hum_latest.loc[:, 'LOS'] = (
    df_hum_latest['LOS'].astype('Int64')
)

df_hum_latest = df_hum_latest[(df_hum_latest['LOS'] >= 0)].copy()
print(df_hum_latest.shape)

# -------------------------------------------------
# Remove extreme savings outliers
# -------------------------------------------------
df_hum_latest = df_hum_latest[
    df_hum_latest['IDSavings'] < 650000
].reset_index(drop=True)

print(df_hum_latest.shape)

# Date only
df_hum_latest['InitialDeterminationDate_date'] = (
    df_hum_latest['InitialDeterminationDate'].dt.date
)

# Year
df_hum_latest['InitialDetermination_Year'] = (
    df_hum_latest['InitialDeterminationDate'].dt.year
)

df_hum_latest["InitialDeterminationDate_date"] = pd.to_datetime(
    df_hum_latest["InitialDeterminationDate_date"],
    errors="coerce"   # handles bad / empty values safely
)


df_hum_latest['InitialDeterminationStatus_Flag'] = (
    df_hum_latest['InitialDeterminationStatus']
    .str.strip()
    .str.upper()
    .map({
        'APPROVED': 0,
        'DENIED': 1
    })
)


df_hum_latest.shape

(76355, 291)
(76355, 291)
<StringArray>
['01', '10']
Length: 2, dtype: string
(76351, 291)
len_adrg
3    75215
4     1136
Name: count, dtype: int64
Group_Type
01    75214
10     1137
Name: count, dtype: Int64
(76340, 292)
Control_ID
HUPTMR-65573    1
HUPTMR-71671    1
HUPTMR-49129    1
HUPRMR-9433     1
HUPRMR-9418     1
Name: count, dtype: Int64
(76340, 293)
(76339, 293)
(76337, 293)
(76337, 293)


(76337, 296)

In [5]:
df_hum_ms_drg = df_hum_latest[df_hum_latest['Group_Type']=='01']
print(df_hum_ms_drg.shape)

df_hum_ms_drg['len_adrg'].value_counts()

(75206, 296)


len_adrg
3    75206
Name: count, dtype: int64

In [6]:
df = df_hum_ms_drg.copy()

adx_cols = [f"A_DX{i}" for i in range(2, 26)]
apx_cols = [f"A_PX{i}" for i in range(1, 26)]


df["A_DX_List"] = df[adx_cols].values.tolist()
df["A_PX_List"] = df[apx_cols].values.tolist()


# Convert to sorted comma separated values
df["A_DX_List"] = df["A_DX_List"].apply(lambda x: ",".join(sorted([str(i) for i in x if pd.notna(i)])))
df["A_PX_List"] = df["A_PX_List"].apply(lambda x: ",".join(sorted([str(i) for i in x if pd.notna(i)])))



def clean_sdx_list(val):
    if pd.isna(val) or val == "":
        return ""
    
    lst = val.split(",")  # split comma-separated values
    cleaned = []
    
    for item in lst:
        item = str(item).upper().strip()       # normalize
        item = item.replace("- MCC", "")       # remove MCC
        item = item.replace("- CC", "")        # remove CC
        item = item.replace("MCC", "")         # safety
        item = item.replace("CC", "")          # safety
        item = item.strip().replace("-", "")   # final cleanup
        cleaned.append(item)
    
    return ",".join(sorted(cleaned))  # sort again and join

df["A_DX_List_Clean"] = df["A_DX_List"].apply(clean_sdx_list)


mccandcclist_df = pd.read_excel(r"C:\Arun_MIX\MCCCCList.xlsx")
mccandcclist_df['ICDCode'] = (
    mccandcclist_df['ICDCode']
    .astype(str)
    .str.strip()
    .str.upper()
)
mccandcclist = dict(zip(mccandcclist_df['ICDCode'], mccandcclist_df['MCCorCC']))
print(f"✅ Loaded MCC/CC list with {len(mccandcclist)} entries.")

def map_sdx_types(icd_string, lookup_dict):

    if pd.isna(icd_string) or icd_string == "":
        return []

    icd_list = icd_string.split(",")

    cleaned_list = []
    for code in icd_list:
        code_clean = str(code).strip().upper()
        tag = lookup_dict.get(code_clean, "")
        cleaned_list.append(f"{code_clean} - {tag}")

    return cleaned_list


df['A_DX_Type_list'] = df['A_DX_List_Clean'].apply(lambda x: map_sdx_types(x, mccandcclist))



def split_dx_types(dx_list):

    mcc = [x.split(" - ")[0] for x in dx_list if "- MCC" in x]
    cc = [x.split(" - ")[0] for x in dx_list if "- CC" in x]
    general = [x.split(" - ")[0] for x in dx_list if "- MCC" not in x and "- CC" not in x]

    return pd.Series({
        "A_DX_MCC_Set": ",".join(sorted(mcc)),
        "A_DX_CC_Set": ",".join(sorted(cc)),
        "A_DX_General_Set": ",".join(sorted(general)),
        "A_DX_MCC_Count": len(mcc),
        "A_DX_CC_Count": len(cc),
        "A_DX_General_Count": len(general)
    })


df[[
    "A_DX_MCC_Set",
    "A_DX_CC_Set",
    "A_DX_General_Set",
    "A_DX_MCC_Count",
    "A_DX_CC_Count",
    "A_DX_General_Count"
]] = df["A_DX_Type_list"].apply(split_dx_types)

#df.head(1)

✅ Loaded MCC/CC list with 17913 entries.


In [7]:
df_with_sdx = df[~(df['A_DX_List'].notna() & (df['A_DX_List'].str.len() == 0))]
print(df_with_sdx.shape)

(73436, 306)


In [8]:
surgical_drg = pd.read_excel(r"C:\Users\arunkumara\Downloads\Surgical DRGs.xlsx",dtype=str)
surgical_drg.head(1)

,DRGCode,MDC,DRGDesc,DRGCost,Version
0,001,MDC P,Heart transplant or implant of heart assist system with MCC,27.0986,41.1


In [9]:
# Ensure string type
df_with_sdx["ADRG"] = df_with_sdx["ADRG"].astype(str)
surgical_drg["DRGCode"] = surgical_drg["DRGCode"].astype(str)

# Create Surgical_DRG flag
df_with_sdx["Surgical_DRG"] = df_with_sdx["ADRG"].isin(surgical_drg["DRGCode"]).astype(int)

C:\Users\arunkumara\AppData\Local\Temp\8\ipykernel_35828\344122554.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_sdx["ADRG"] = df_with_sdx["ADRG"].astype(str)
C:\Users\arunkumara\AppData\Local\Temp\8\ipykernel_35828\344122554.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_sdx["Surgical_DRG"] = df_with_sdx["ADRG"].isin(surgical_drg["DRGCode"]).astype(int)


In [10]:
cols_to_keep = [
    "Claim_Number",
    "PaymentType",
    "Payments",
    "Tax_ID",
    "Provider_ID",
    "HospitalName",
    "LOS",
    "AGE",
    "ADRG",
    "Surgical_DRG",
    "PRIM_DX",
    "A_DX_MCC_Set",
    "A_DX_CC_Set",
    "A_DX_General_Set",
    "A_DX_MCC_Count",
    "A_DX_CC_Count",
    "A_DX_General_Count",
    "A_PX_List",
    "InitialDeterminationStatus"
]

df_new = df_with_sdx[cols_to_keep].copy()

df_new["TARGET"] = df_new["InitialDeterminationStatus"].map({
    "APPROVED": 0,
    "DENIED": 1
})
df_new.head()

,Claim_Number,PaymentType,Payments,Tax_ID,Provider_ID,HospitalName,LOS,AGE,ADRG,Surgical_DRG,PRIM_DX,A_DX_MCC_Set,A_DX_CC_Set,A_DX_General_Set,A_DX_MCC_Count,A_DX_CC_Count,A_DX_General_Count,A_PX_List,InitialDeterminationStatus,TARGET
0,820252450252252,PostPay,15464.79,455055149,455055149A,ST ANTHONY SHAWNEE HOSPITAL IN,7,52,616,1,E11621,"E1110,G928,G9341","F3181,L02611,L97419,M86171,M869,R45851","B1920,E039,E1140,E1165,E88819,F1010,F1511,F209,I10,L97519,Z5900,Z5941,Z5982,Z5986,Z794",3,6,15,"0JBQ0ZZ,0KBV0ZZ,0Y6R0Z0",APPROVED,0
1,820253180530927,PostPay,11699.99,566017737,566017737F,WAKEMED RALEIGH CAMPUS,7,85,193,0,J189,"I5033,J9601",E873,"E039,E1165,E785,F0394,F419,I110,K219,K5900,K7469,Z1152,Z66,Z794,Z79890,Z79899,Z87440",2,1,15,"5A0945A,B32T1ZZ",APPROVED,0
2,820251200832345,PostPay,15543.60,900054201,900054201,BANNER DESERT MEDICAL,6,71,377,0,K921,"J9601,N186","D62,I132,I5032,Z6841","D631,E1122,E6601,E66813,E785,E8351,G4733,I2510,I2720,I4891,J449,Z8673,Z955,Z992",2,4,14,"5A09357,5A1D70Z",APPROVED,0
4,820253280447437,PostPay,15628.23,160743102,160743102A,OLEAN GENERAL HOSPITAL,10,67,871,0,A419,"G9341,J189,J690,J9601,R532,R6521","E870,R64","D696,D72819,E119,E7800,E8339,E860,E861,E876,F0280,F79,G20A1,G40909,I959,J9809,N3281,R4182",6,2,16,"0BH17EZ,5A09357,5A1945Z",APPROVED,0
5,820251920785407,PostPay,11594.74,592142859,000198453,SHANDS JACKSONVILLE MEDICAL CE,4,90,200,0,J95811,,"C3412,E440,E871","E1140,E1165,E7800,F17210,J449,N3020,N401,R338,R54,Z6821,Z7984,Z79899,Z8546,Z85828,Z86718,Z87442",0,3,16,"0BBG3ZX,0W9B30Z",APPROVED,0


In [23]:
# ============================================================
# 🔥 FINAL FEATURE ENGINEERING (ALIGNED TO YOUR COLUMNS)
# ============================================================

import pandas as pd
import numpy as np

df = df_new.copy()

# -------------------------------
# 0. TARGET SAFETY
# -------------------------------
if "TARGET" not in df.columns:
    df["TARGET"] = df["InitialDeterminationStatus"].map({
        "APPROVED": 0,
        "DENIED": 1
    })

# -------------------------------
# 1. BASIC DERIVED FEATURES
# -------------------------------

# Procedure count from A_PX_List
df["proc_count"] = df["A_PX_List"].apply(
    lambda x: 0 if pd.isna(x) or x == "" else len(x.split(","))
)

# Total SDX count
df["total_dx_count"] = (
    df["A_DX_MCC_Count"] +
    df["A_DX_CC_Count"] +
    df["A_DX_General_Count"]
)

# Flags
df["has_mcc"] = (df["A_DX_MCC_Count"] > 0).astype(int)
df["has_cc"]  = (df["A_DX_CC_Count"] > 0).astype(int)

# Severity score
df["severity_score"] = 0
df.loc[df["has_cc"] == 1, "severity_score"] = 1
df.loc[df["has_mcc"] == 1, "severity_score"] = 2

# -------------------------------
# 2. PDX–DRG RELATION
# -------------------------------
pdx_drg_counts = (
    df.groupby(["PRIM_DX", "ADRG"])
      .size()
      .reset_index(name="pdx_drg_count")
)

df = df.merge(pdx_drg_counts, on=["PRIM_DX","ADRG"], how="left")
df["pdx_drg_count"] = df["pdx_drg_count"].fillna(0)

df["is_rare_pdx_drg"] = (df["pdx_drg_count"] <= 5).astype(int)
df["is_valid_pdx_drg"] = (df["pdx_drg_count"] >= 6).astype(int)

# -------------------------------
# 3. DENIAL PATTERN (PDX + DRG)
# -------------------------------
pdx_drg_status = (
    df.groupby(["PRIM_DX","ADRG","TARGET"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

pdx_drg_status = pdx_drg_status.rename(columns={
    0: "approved_count",
    1: "denied_count"
})

pdx_drg_status["total_count"] = (
    pdx_drg_status["approved_count"] +
    pdx_drg_status["denied_count"]
)

df = df.merge(pdx_drg_status, on=["PRIM_DX","ADRG"], how="left").fillna(0)

df["denial_rate"] = df["denied_count"] / (df["total_count"] + 1)

In [24]:
# ============================================================
# 🔥 MCC CODE LEVEL RELATION (NOT has_mcc)
# ============================================================

# -------------------------------
# 1. EXPLODE MCC CODES
# -------------------------------
df_mcc = df[["PRIM_DX","ADRG","TARGET","A_DX_MCC_Set"]].copy()

df_mcc["MCC_CODE"] = df_mcc["A_DX_MCC_Set"].str.split(",")
df_mcc = df_mcc.explode("MCC_CODE")

# clean
df_mcc["MCC_CODE"] = df_mcc["MCC_CODE"].astype(str).str.strip()
df_mcc = df_mcc[df_mcc["MCC_CODE"] != ""]

# -------------------------------
# 2. MCC CODE + DRG COUNT
# -------------------------------
mcc_code_counts = (
    df_mcc.groupby(["MCC_CODE","ADRG"])
    .size()
    .reset_index(name="mcc_code_drg_count")
)

# -------------------------------
# 3. MCC CODE DENIAL PATTERN
# -------------------------------
mcc_code_status = (
    df_mcc.groupby(["MCC_CODE","ADRG","TARGET"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

mcc_code_status = mcc_code_status.rename(columns={
    0: "mcc_code_approved",
    1: "mcc_code_denied"
})

mcc_code_status["mcc_code_total"] = (
    mcc_code_status["mcc_code_approved"] +
    mcc_code_status["mcc_code_denied"]
)

mcc_code_status["mcc_code_denial_rate"] = (
    mcc_code_status["mcc_code_denied"] /
    (mcc_code_status["mcc_code_total"] + 1)
)

# -------------------------------
# 4. MERGE BACK (AGGREGATE PER ROW)
# -------------------------------
df_mcc = df_mcc.merge(
    mcc_code_counts,
    on=["MCC_CODE","ADRG"],
    how="left"
)

df_mcc = df_mcc.merge(
    mcc_code_status,
    on=["MCC_CODE","ADRG"],
    how="left"
)

# -------------------------------
# 5. AGGREGATE BACK TO CLAIM LEVEL
# -------------------------------
mcc_agg = (
    df_mcc.groupby(df_mcc.index)
    .agg({
        "mcc_code_drg_count": "mean",
        "mcc_code_denial_rate": "mean"
    })
    .reset_index(drop=True)
)

df["mcc_code_avg_count"] = mcc_agg["mcc_code_drg_count"].fillna(0)
df["mcc_code_avg_denial"] = mcc_agg["mcc_code_denial_rate"].fillna(0)

# rare flag
df["mcc_code_rare"] = (df["mcc_code_avg_count"] <= 5).astype(int)

In [25]:
# ============================================================
# 🔥 CC CODE LEVEL RELATION
# ============================================================

df_cc = df[["PRIM_DX","ADRG","TARGET","A_DX_CC_Set"]].copy()

df_cc["CC_CODE"] = df_cc["A_DX_CC_Set"].str.split(",")
df_cc = df_cc.explode("CC_CODE")

df_cc["CC_CODE"] = df_cc["CC_CODE"].astype(str).str.strip()
df_cc = df_cc[df_cc["CC_CODE"] != ""]

cc_counts = (
    df_cc.groupby(["CC_CODE","ADRG"])
    .size()
    .reset_index(name="cc_code_drg_count")
)

df_cc = df_cc.merge(cc_counts, on=["CC_CODE","ADRG"], how="left")

cc_agg = (
    df_cc.groupby(df_cc.index)
    .agg({"cc_code_drg_count": "mean"})
    .reset_index(drop=True)
)

df["cc_code_avg_count"] = cc_agg["cc_code_drg_count"].fillna(0)
df["cc_code_rare"] = (df["cc_code_avg_count"] <= 5).astype(int)

In [26]:
df.head()

,Claim_Number,PaymentType,Payments,Tax_ID,Provider_ID,HospitalName,LOS,AGE,ADRG,Surgical_DRG,PRIM_DX,A_DX_MCC_Set,A_DX_CC_Set,A_DX_General_Set,A_DX_MCC_Count,A_DX_CC_Count,A_DX_General_Count,A_PX_List,InitialDeterminationStatus,TARGET,proc_count,total_dx_count,has_mcc,has_cc,severity_score,pdx_drg_count,is_rare_pdx_drg,is_valid_pdx_drg,approved_count,denied_count,total_count,denial_rate,mcc_code_avg_count,mcc_code_avg_denial,mcc_code_rare,cc_code_avg_count,cc_code_rare
0,820252450252252,PostPay,15464.79,455055149,455055149A,ST ANTHONY SHAWNEE HOSPITAL IN,7,52,616,1,E11621,"E1110,G928,G9341","F3181,L02611,L97419,M86171,M869,R45851","B1920,E039,E1140,E1165,E88819,F1010,F1511,F209,I10,L97519,Z5900,Z5941,Z5982,Z5986,Z794",3,6,15,"0JBQ0ZZ,0KBV0ZZ,0Y6R0Z0",APPROVED,0,3,24,1,1,2,15,0,1,15,0,15,0.000000,3.0,0.250000,1,1.0,1
1,820253180530927,PostPay,11699.99,566017737,566017737F,WAKEMED RALEIGH CAMPUS,7,85,193,0,J189,"I5033,J9601",E873,"E039,E1165,E785,F0394,F419,I110,K219,K5900,K7469,Z1152,Z66,Z794,Z79890,Z79899,Z87440",2,1,15,"5A0945A,B32T1ZZ",APPROVED,0,2,18,1,1,2,1733,0,1,1528,205,1733,0.118224,11.0,0.166667,0,7.0,0
2,820251200832345,PostPay,15543.60,900054201,900054201,BANNER DESERT MEDICAL,6,71,377,0,K921,"J9601,N186","D62,I132,I5032,Z6841","D631,E1122,E6601,E66813,E785,E8351,G4733,I2510,I2720,I4891,J449,Z8673,Z955,Z992",2,4,14,"5A09357,5A1D70Z",APPROVED,0,2,20,1,1,2,208,0,1,197,11,208,0.052632,19.0,0.200000,0,4.0,1
3,820253280447437,PostPay,15628.23,160743102,160743102A,OLEAN GENERAL HOSPITAL,10,67,871,0,A419,"G9341,J189,J690,J9601,R532,R6521","E870,R64","D696,D72819,E119,E7800,E8339,E860,E861,E876,F0280,F79,G20A1,G40909,I959,J9809,N3281,R4182",6,2,16,"0BH17EZ,5A09357,5A1945Z",APPROVED,0,3,24,1,1,2,3830,0,1,2341,1489,3830,0.388671,462.0,0.069114,0,19.0,0
4,820251920785407,PostPay,11594.74,592142859,000198453,SHANDS JACKSONVILLE MEDICAL CE,4,90,200,0,J95811,,"C3412,E440,E871","E1140,E1165,E7800,F17210,J449,N3020,N401,R338,R54,Z6821,Z7984,Z79899,Z8546,Z85828,Z86718,Z87442",0,3,16,"0BBG3ZX,0W9B30Z",APPROVED,0,2,19,0,1,1,37,0,1,37,0,37,0.000000,1123.0,0.146797,0,17.0,0


In [ ]:


# -------------------------------
# 4. MCC/CC + DRG RELATION
# -------------------------------
mcc_drg_counts = (
    df.groupby(["PRIM_DX","ADRG","has_mcc","has_cc"])
      .size()
      .reset_index(name="mcc_drg_count")
)

df = df.merge(
    mcc_drg_counts,
    on=["PRIM_DX","ADRG","has_mcc","has_cc"],
    how="left"
).fillna(0)

df["mcc_drg_rare"] = (df["mcc_drg_count"] <= 5).astype(int)

# MCC denial
mcc_status = (
    df.groupby(["PRIM_DX","ADRG","has_mcc","has_cc","TARGET"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

mcc_status = mcc_status.rename(columns={
    0: "mcc_approved",
    1: "mcc_denied"
})

mcc_status["mcc_total"] = (
    mcc_status["mcc_approved"] +
    mcc_status["mcc_denied"]
)

df = df.merge(
    mcc_status,
    on=["PRIM_DX","ADRG","has_mcc","has_cc"],
    how="left"
).fillna(0)

df["mcc_denial_rate"] = df["mcc_denied"] / (df["mcc_total"] + 1)

# -------------------------------
# 5. SURGICAL + PROCEDURE LOGIC
# -------------------------------

# Mismatch
df["proc_mismatch"] = (
    (df["Surgical_DRG"] == 1) & (df["proc_count"] == 0)
).astype(int)

# Excess procedure
df["excess_proc_flag"] = (
    (df["Surgical_DRG"] == 0) & (df["proc_count"] >= 3)
).astype(int)

# Surgical frequency
surg_counts = (
    df.groupby(["PRIM_DX","ADRG","Surgical_DRG"])
      .size()
      .reset_index(name="surg_drg_count")
)

df = df.merge(
    surg_counts,
    on=["PRIM_DX","ADRG","Surgical_DRG"],
    how="left"
).fillna(0)

df["surg_drg_rare"] = (df["surg_drg_count"] <= 5).astype(int)

# Surgical denial
surg_status = (
    df.groupby(["PRIM_DX","ADRG","Surgical_DRG","TARGET"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

surg_status = surg_status.rename(columns={
    0: "surg_approved",
    1: "surg_denied"
})

surg_status["surg_total"] = (
    surg_status["surg_approved"] +
    surg_status["surg_denied"]
)

df = df.merge(
    surg_status,
    on=["PRIM_DX","ADRG","Surgical_DRG"],
    how="left"
).fillna(0)

df["surg_denial_rate"] = df["surg_denied"] / (df["surg_total"] + 1)

# -------------------------------
# 6. LOS + AGE NORMALIZATION
# -------------------------------
df["los_zscore"] = (
    (df["LOS"] - df.groupby("ADRG")["LOS"].transform("mean")) /
    (df.groupby("ADRG")["LOS"].transform("std") + 1)
)

df["age_zscore"] = (
    (df["AGE"] - df.groupby("ADRG")["AGE"].transform("mean")) /
    (df.groupby("ADRG")["AGE"].transform("std") + 1)
)



In [22]:
df.head()

,Claim_Number,PaymentType,Payments,Tax_ID,Provider_ID,HospitalName,LOS,AGE,ADRG,Surgical_DRG,PRIM_DX,A_DX_MCC_Set,A_DX_CC_Set,A_DX_General_Set,A_DX_MCC_Count,A_DX_CC_Count,A_DX_General_Count,A_PX_List,InitialDeterminationStatus,TARGET,proc_count,total_dx_count,has_mcc,has_cc,severity_score,pdx_drg_count,is_rare_pdx_drg,is_valid_pdx_drg,approved_count,denied_count,total_count,denial_rate,mcc_drg_count,mcc_drg_rare,mcc_approved,mcc_denied,mcc_total,mcc_denial_rate,proc_mismatch,excess_proc_flag,surg_drg_count,surg_drg_rare,surg_approved,surg_denied,surg_total,surg_denial_rate,los_zscore,age_zscore
0,820252450252252,PostPay,15464.79,455055149,455055149A,ST ANTHONY SHAWNEE HOSPITAL IN,7,52,616,1,E11621,"E1110,G928,G9341","F3181,L02611,L97419,M86171,M869,R45851","B1920,E039,E1140,E1165,E88819,F1010,F1511,F209,I10,L97519,Z5900,Z5941,Z5982,Z5986,Z794",3,6,15,"0JBQ0ZZ,0KBV0ZZ,0Y6R0Z0",APPROVED,0,3,24,1,1,2,15,0,1,15,0,15,0.000000,15,0,15,0,15,0.000000,0,0,15,0,15,0,15,0.000000,-1.09631,-1.092522
1,820253180530927,PostPay,11699.99,566017737,566017737F,WAKEMED RALEIGH CAMPUS,7,85,193,0,J189,"I5033,J9601",E873,"E039,E1165,E785,F0394,F419,I110,K219,K5900,K7469,Z1152,Z66,Z794,Z79890,Z79899,Z87440",2,1,15,"5A0945A,B32T1ZZ",APPROVED,0,2,18,1,1,2,1733,0,1,1528,205,1733,0.118224,1656,0,1467,189,1656,0.114062,0,0,1733,0,1528,205,1733,0.118224,-0.11094,0.877579
2,820251200832345,PostPay,15543.60,900054201,900054201,BANNER DESERT MEDICAL,6,71,377,0,K921,"J9601,N186","D62,I132,I5032,Z6841","D631,E1122,E6601,E66813,E785,E8351,G4733,I2510,I2720,I4891,J449,Z8673,Z955,Z992",2,4,14,"5A09357,5A1D70Z",APPROVED,0,2,20,1,1,2,208,0,1,197,11,208,0.052632,206,0,195,11,206,0.053140,0,0,208,0,197,11,208,0.052632,-0.258782,-0.214177
3,820253280447437,PostPay,15628.23,160743102,160743102A,OLEAN GENERAL HOSPITAL,10,67,871,0,A419,"G9341,J189,J690,J9601,R532,R6521","E870,R64","D696,D72819,E119,E7800,E8339,E860,E861,E876,F0280,F79,G20A1,G40909,I959,J9809,N3281,R4182",6,2,16,"0BH17EZ,5A09357,5A1945Z",APPROVED,0,3,24,1,1,2,3830,0,1,2341,1489,3830,0.388671,3760,0,2318,1442,3760,0.383409,0,1,3830,0,2341,1489,3830,0.388671,-0.104555,-0.440909
4,820251920785407,PostPay,11594.74,592142859,000198453,SHANDS JACKSONVILLE MEDICAL CE,4,90,200,0,J95811,,"C3412,E440,E871","E1140,E1165,E7800,F17210,J449,N3020,N401,R338,R54,Z6821,Z7984,Z79899,Z8546,Z85828,Z86718,Z87442",0,3,16,"0BBG3ZX,0W9B30Z",APPROVED,0,2,19,0,1,1,37,0,1,37,0,37,0.000000,37,0,37,0,37,0.000000,0,0,37,0,37,0,37,0.000000,0.064851,1.701919


In [ ]:
# -------------------------------
# 7. FINAL FEATURES
# -------------------------------
features = [
    "ADRG","PRIM_DX","Surgical_DRG",

    "A_DX_MCC_Set","A_DX_CC_Set","A_DX_General_Set"

    "LOS","AGE","los_zscore","age_zscore",

    "A_DX_MCC_Count","A_DX_CC_Count","A_DX_General_Count",
    "has_mcc","has_cc","severity_score",

    "proc_count","proc_mismatch","excess_proc_flag",

    "pdx_drg_count","is_rare_pdx_drg",
    "denial_rate",

    "mcc_drg_count","mcc_drg_rare","mcc_denial_rate",

    "surg_drg_count","surg_drg_rare","surg_denial_rate"
]

print(df[features].head(10))

# prev

In [19]:
# ============================================================
# 🔥 FULL FEATURE ENGINEERING: PDX (PRIM_DX) + DRG (ADRG)
# ============================================================

import pandas as pd
import numpy as np


df = df_hum_ms_drg.copy()
# -------------------------------
# 0. STANDARDIZE TARGET
# -------------------------------
df["TARGET"] = df["InitialDeterminationStatus"].map({
    "APPROVED": 0,
    "DENIED": 1
})

# -------------------------------
# 1. PDX–DRG COUNT
# -------------------------------
pdx_drg_counts = (
    df.groupby(["PRIM_DX", "ADRG"])
      .size()
      .reset_index(name="pdx_drg_count")
)

df = df.merge(pdx_drg_counts, on=["PRIM_DX", "ADRG"], how="left")
df["pdx_drg_count"] = df["pdx_drg_count"].fillna(0).astype(int)

# -------------------------------
# 2. RARE / VALID FLAGS
# -------------------------------
df["is_rare_pdx_drg"] = (df["pdx_drg_count"] <= 5).astype(int)
df["is_valid_pdx_drg"] = (df["pdx_drg_count"] >= 6).astype(int)

# -------------------------------
# 3. FREQUENCY BUCKET
# -------------------------------
def bucket_count(x):
    if x == 0:
        return "unseen"
    elif x <= 5:
        return "rare"
    elif x <= 20:
        return "low"
    elif x <= 100:
        return "medium"
    else:
        return "high"

df["pdx_drg_freq_bucket"] = df["pdx_drg_count"].apply(bucket_count)

freq_map = {
    "unseen": 0,
    "rare": 1,
    "low": 2,
    "medium": 3,
    "high": 4
}

df["pdx_drg_freq_encoded"] = df["pdx_drg_freq_bucket"].map(freq_map)

# -------------------------------
# 4. STRENGTH (NORMALIZED)
# -------------------------------
pdx_total = df.groupby("PRIM_DX")["pdx_drg_count"].transform("sum")
df["pdx_drg_strength"] = df["pdx_drg_count"] / (pdx_total + 1)

# -------------------------------
# 5. APPROVED / DENIED COUNTS
# -------------------------------
pdx_drg_status = (
    df.groupby(["PRIM_DX", "ADRG", "TARGET"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

pdx_drg_status.columns.name = None
pdx_drg_status = pdx_drg_status.rename(columns={
    0: "approved_count",
    1: "denied_count"
})

# Ensure both columns exist
for col in ["approved_count", "denied_count"]:
    if col not in pdx_drg_status.columns:
        pdx_drg_status[col] = 0

pdx_drg_status["total_count"] = (
    pdx_drg_status["approved_count"] +
    pdx_drg_status["denied_count"]
)

# Merge back
df = df.merge(
    pdx_drg_status,
    on=["PRIM_DX", "ADRG"],
    how="left"
)

df[["approved_count", "denied_count", "total_count"]] = \
    df[["approved_count", "denied_count", "total_count"]].fillna(0)

# -------------------------------
# 6. RATES + FLAGS
# -------------------------------
df["denial_rate"] = df["denied_count"] / (df["total_count"] + 1)
df["approval_rate"] = df["approved_count"] / (df["total_count"] + 1)

df["high_denial_flag"] = (df["denial_rate"] >= 0.3).astype(int)

# -------------------------------
# 7. FINAL FEATURES LIST
# -------------------------------
features = [
    "ADRG",
    "PRIM_DX",
    "pdx_drg_count",
    "is_rare_pdx_drg",
    "is_valid_pdx_drg",
    "pdx_drg_freq_encoded",
    "pdx_drg_strength",
    "approved_count",
    "denied_count",
    "denial_rate",
    "approval_rate",
    "high_denial_flag"
]

# -------------------------------
# 8. VIEW RESULT
# -------------------------------
df[features].head(10)

,ADRG,PRIM_DX,pdx_drg_count,is_rare_pdx_drg,is_valid_pdx_drg,pdx_drg_freq_encoded,pdx_drg_strength,approved_count,denied_count,denial_rate,approval_rate,high_denial_flag
0,616,E11621,15,0,1,2,0.004732,15,0,0.000000,0.937500,0
1,193,J189,1754,0,1,4,0.000559,1546,208,0.118519,0.880912,0
2,377,K921,213,0,1,4,0.004350,202,11,0.051402,0.943925,0
3,871,A419,3985,0,1,4,0.000239,2472,1513,0.379579,0.620171,1
4,200,J95811,37,0,1,3,0.008569,37,0,0.000000,0.973684,0
5,811,D5700,34,0,1,3,0.029285,34,0,0.000000,0.971429,0
6,683,N179,4039,0,1,4,0.000239,3500,539,0.133416,0.866337,0
7,948,G893,8,0,1,2,0.033473,8,0,0.000000,0.888889,0
8,683,N179,4039,0,1,4,0.000239,3500,539,0.133416,0.866337,0
9,465,T8453XA,7,0,1,2,0.005728,7,0,0.000000,0.875000,0
